# BK 879B LCR Meter — Driver Test Notebook

Standalone test of `BK879BDriver` — bypasses the workspace component, the station wrapper, the device bus, and the orchestrator UI. Use this to confirm the **raw hardware comms** work in isolation when diagnosing "is the meter even responding?" issues.

**Prereqs**
- Meter plugged in via USB.
- Meter in **RMT** mode (press the USB button on the front panel until the RMT indicator shows).
- Jupyter running with the same Python that has `pyserial` + the workspace package available. If permission errors hit when opening `/dev/ttyUSB*`, launch Jupyter as `sudo` or add yourself to the `dialout` group.

Run cells top to bottom.

## 1. Imports

In [1]:
import sys
from pathlib import Path

# Make the workspace package importable when running from the project
# folder without a global pip install.
_REPO_ROOT = Path('/home/dorna/Downloads/workspace/workspace')
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from workspace.components.multi_meter.bk879b_driver import (
    BK879B,
    Measurement,
    list_ports,
    format_value,
    format_frequency,
)

## 2. List candidate serial ports

Anything flagged `likely=True` matches the FTDI / CP210x USB-serial chip families the BK 879B ships with. The one with the meter plugged in should appear here.

In [2]:
for p in list_ports():
    tag = ' ← likely BK 879B' if p['likely'] else ''
    print(f"{p['port']:24s} {p['description']!r:50s}{tag}")

/dev/ttyUSB1             'CP2102 USB to UART Bridge Controller - CP2102 USB to UART Bridge Controller' ← likely BK 879B
/dev/ttyAMA10            'ttyAMA10'                                        


## 3. Connect

Use the stable `/dev/serial/by-id/...` path so the test still works after reboot or replug. Set `PORT = None` to auto-detect by USB VID instead.

After `connect()`, `meter.port` reflects the actual resolved device path.

In [3]:
PORT = '/dev/serial/by-id/usb-Silicon_Labs_CP2102_USB_to_UART_Bridge_Controller_0001-if00-port0'
# PORT = None  # uncomment to auto-detect by USB VID

meter = BK879B(port=PORT)
print(f'is_connected (before): {meter.is_connected()}')

opened = meter.connect()
print(f'connect() returned: {opened}')
print(f'meter.port (resolved): {meter.port}')
print(f'is_connected (after):  {meter.is_connected()}')

is_connected (before): False
connect() returned: True
meter.port (resolved): /dev/serial/by-id/usb-Silicon_Labs_CP2102_USB_to_UART_Bridge_Controller_0001-if00-port0
is_connected (after):  True


## 4. IDN handshake

`check_connection()` sends `*IDN?` and returns whether the meter responds. If `ok=False` here, the most common cause is the meter not being in RMT mode.

In [4]:
ok, idn = meter.check_connection()
print(f'check_connection → ok={ok}, idn={idn!r}')

check_connection → ok=False, idn=None


## 5. Initialize

Same IDN handshake but raises a `RuntimeError` if no response — useful when you want the cell to fail loudly instead of silently returning `False`.

In [5]:
resp = meter.initialize()
print(f'initialize() → {resp!r}')

initialize() → '879B LCR Meter,VER2.1.1102,SN127J23128'


## 6. Read capacitance

Returns a `Measurement` with `primary`, `primary_unit`, `secondary` (dissipation factor D), `function`, `frequency`, `raw`. Connect a small known capacitor across the test leads — a 1 nF reference cap is a good starter target.

In [8]:
m = meter.read_capacitance(mode='Cp', frequency=1000)
print(f'C: {format_value(m.primary, m.primary_unit)}')
print(f'D: {m.secondary:.4g}')
print(f'@ {format_frequency(m.frequency)}')
print(f'raw: {m.raw!r}')

C: 2.032 pF
D: 0
@ 1 kHz
raw: '+2.03233e-12,+0.00000e+00,N'


## 7. Read inductance

In [ ]:
m = meter.read_inductance(mode='Ls', frequency=1000)
print(f'L: {format_value(m.primary, m.primary_unit)}')
print(f'Q: {m.secondary:.4g}')
print(f'@ {format_frequency(m.frequency)}')
print(f'raw: {m.raw!r}')

## 8. Read resistance

In [ ]:
m = meter.read_resistance(frequency=1000)
print(f'R: {format_value(m.primary, m.primary_unit)}')
print(f'@ {format_frequency(m.frequency)}')
print(f'raw: {m.raw!r}')

## 9. Read impedance

In [ ]:
m = meter.read_impedance(frequency=1000)
print(f'Z: {format_value(m.primary, m.primary_unit)}')
print(f'@ {format_frequency(m.frequency)}')
print(f'raw: {m.raw!r}')

## 10. Frequency sweep — sanity check

BK 879B supports 100, 120, 1000, and 10000 Hz. Sweeping all four catches frequency-dependent oddities (loose leads, fixture parasitics).

In [ ]:
for freq in (100, 120, 1000, 10000):
    m = meter.read_capacitance(mode='Cp', frequency=freq)
    print(f'{format_frequency(m.frequency):>8s}  '
          f'C={format_value(m.primary, m.primary_unit):>10s}  '
          f'D={m.secondary:.4g}')

## 11. Cleanup

`close()` sends `*GTL` (go to local) so the front panel is usable again, then releases the serial port.

In [ ]:
meter.close()
print(f'is_connected after close: {meter.is_connected()}')